##### Multilayer perceptron

creating subclass of Module

In [ ]:
import torch
class NeuralNetwork(torch.nn.Module): #create from module to inherit pytorch functionality
    def __init__(self, num_inputs, num_outputs): #constructor, takes in number of inputs and outputs for the network
        super().__init__() 

        self.layers = torch.nn.Sequential ( #allows us to stack layers in order with data flow easily.
            #layer 1 
            torch.nn.Linear(num_inputs, 30), #30 neurons each with weights and bias, Liner: y = weighted sum of inputs + bias
            torch.nn.ReLU(), #activation functions, ReLU: y = max(0,x) if neg ignore

            #layer 2
            torch.nn.Linear(30, 20),
            torch.nn.ReLU(),

            #output layer
            torch.nn.Linear(20, num_outputs) #produces desired outputs for prediction 
        )
    
    #take input x and push through layers to get output
    def forward(self,x): #how the data flows 
        logits = self.layers(x) #pass data through layers
        return logits
    #remember backpropagation is done by pytorch automatically 

Explore architecture + params

In [5]:
model = NeuralNetwork(50,3)
print(model) #gives architecture of the model

#check total number of trainable parameters
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad) #counting number of parameters with gradient tracking
print(f"Total trainable parameters: {num_params}")

NeuralNetwork(
  (layers): Sequential(
    (0): Linear(in_features=50, out_features=30, bias=True)
    (1): ReLU()
    (2): Linear(in_features=30, out_features=20, bias=True)
    (3): ReLU()
    (4): Linear(in_features=20, out_features=3, bias=True)
  )
)
Total trainable parameters: 2213


In [ ]:
print(model.layers[0].weight)#looking at layer 1 to see weights of the neurons 
#these are the starting weights before training 

print(model.layers[0].weight.shape)
#matches our function 30 x 50(input) 

Parameter containing:
tensor([[-0.1232, -0.0388,  0.0753,  ..., -0.1202,  0.0928, -0.1392],
        [ 0.1046,  0.0444,  0.0719,  ..., -0.1164, -0.0814, -0.1188],
        [ 0.0734, -0.0021,  0.0820,  ..., -0.1231,  0.0604, -0.0664],
        ...,
        [ 0.1007, -0.0565,  0.0704,  ..., -0.1343,  0.1185, -0.0334],
        [-0.0775,  0.0161, -0.0782,  ...,  0.1152,  0.0504, -0.0733],
        [ 0.1106,  0.0210,  0.1082,  ..., -0.0939, -0.0422,  0.0725]],
       requires_grad=True)
torch.Size([30, 50])


A challenge is reproducibility. The weights are random numbers every time which is crucial or else all the neurons learn the same thing. 

*however* this makes it hard to debug

So similar to minecraft there are seeds. you get different starting weights but uses a **specfic** sequence of random numbers. 



In [8]:
torch.manual_seed(123)
model = NeuralNetwork(50,3)
print(model.layers[0].weight) 

Parameter containing:
tensor([[-0.0577,  0.0047, -0.0702,  ...,  0.0222,  0.1260,  0.0865],
        [ 0.0502,  0.0307,  0.0333,  ...,  0.0951,  0.1134, -0.0297],
        [ 0.1077, -0.1108,  0.0122,  ...,  0.0108, -0.1049, -0.1063],
        ...,
        [-0.0787,  0.1259,  0.0803,  ...,  0.1218,  0.1303, -0.1351],
        [ 0.1359,  0.0175, -0.0673,  ...,  0.0674,  0.0676,  0.1058],
        [ 0.0790,  0.1343, -0.0293,  ...,  0.0344, -0.0971, -0.0509]],
       requires_grad=True)


Running the model:

When we call the model it would automatically fun the forward pass 

while flowing forward pytorch *builds* a receipt of all the operations **COMPUTATION GRAPH**. Where every output tensor caries a "grad_fn" tag saying I was produced by this operation. 

*tag -> pytorch internal book keeping*




In [10]:
torch.manual_seed(123)
X = torch.randn(1,50) #create random input data 
out = model(X)#pass data through model to get output
print(out)

tensor([[-0.1080,  0.0924,  0.0775]], grad_fn=<AddmmBackward0>)


When the network is already trained and now need to make predictions and just want to run thourgh without building the computation graph: 

In [11]:
with torch.no_grad(): #turn off gradient tracking
    out = model(X)
print(out)

tensor([[-0.1080,  0.0924,  0.0775]])


To convert the raw output(logits) into probabilites. 

use softmax to make it 1.0 scale. 

In [ ]:
with torch.no_grad():
    out = torch.softmax(model(X), dim=1) #turn logits into probabilities
print(out)

#the values a relatively similar because the model is untrained for a initalied random values. 

tensor([[0.2919, 0.3567, 0.3514]])
